# Deteccion de animal escondido en fotos ya confirmadas como planta

Corre BioCLIP (zero-shot, igual mecanica que `Clasificador_PlantaAnimal.ipynb` pero con un modelo entrenado para especies, no CLIP generico) sobre las fotos que YA tienen especie de planta confirmada, buscando un segundo sujeto (ave o insecto) en el cuadro que el escaneo original no capturo.

**No busca la direccion contraria** (planta escondida en fotos de animal): decision de Remi (15-sep), todas las plantas del campus ya se identificaron desde el escaneo inicial, encontrar mas evidencia fotografica de un arbol ya conocido no aporta.

**No duplica fotos todavia.** Solo registra el resultado (scores de insecto/ave/nada) en una tabla nueva, `deteccion_animal_en_planta`. La duplicacion real (mismo `ruta_original`, `ruta_actual`/`especie_id` distintos) es un paso manual posterior, una vez que se decida el umbral de confianza con el que vale la pena confiar en la deteccion.

**Benchmark real (15-sep, esta maquina, sin GPU, 2 hilos de CPU, contra Y:\ directo):** ~1.08s lectura+preprocesamiento, ~0.46s inferencia, ~1.54s/foto total. Con ~2,697 fotos confirmadas, la corrida completa toma aproximadamente una hora.

In [ ]:
import sqlite3
from pathlib import Path

import torch
import open_clip
from PIL import Image

RUTA_BD = Path("../../experimentos/bd/campuslab_volumen.db")
con = sqlite3.connect(RUTA_BD)
print(f"Base de datos: {RUTA_BD.resolve()} ({'existe' if RUTA_BD.exists() else 'NO ENCONTRADA'})")

con.execute("""
    CREATE TABLE IF NOT EXISTS deteccion_animal_en_planta (
        foto_id INTEGER PRIMARY KEY,
        score_insecto REAL,
        score_ave REAL,
        score_nada REAL,
        fecha TEXT DEFAULT CURRENT_TIMESTAMP
    )
""")
con.commit()

print("Cargando BioCLIP...")
modelo, _, preprocess = open_clip.create_model_and_transforms("hf-hub:imageomics/bioclip")
tokenizer = open_clip.get_tokenizer("hf-hub:imageomics/bioclip")
modelo.eval()

PROMPTS = ["a photo of an insect", "a photo of a bird", "a photo of a plant with no animal"]
with torch.no_grad():
    texto_features = modelo.encode_text(tokenizer(PROMPTS))
    texto_features /= texto_features.norm(dim=-1, keepdim=True)
print("Modelo listo.")

In [ ]:
candidatas = con.execute("""
    SELECT id, ruta_actual FROM fotos
    WHERE estado='usable' AND especie_id IS NOT NULL AND ruta_actual NOT LIKE '%.db'
      AND id NOT IN (SELECT foto_id FROM deteccion_animal_en_planta)
""").fetchall()
print(f"Candidatas (confirmadas, sin detectar todavia): {len(candidatas):,}")

In [ ]:
EJECUTAR_REAL = False  # <-- cambialo a True para correr de verdad

if not EJECUTAR_REAL:
    print("EJECUTAR_REAL esta en False. No se proceso nada todavia.")
else:
    procesadas = 0
    errores = 0
    for i, (foto_id, ruta) in enumerate(candidatas, 1):
        try:
            img = Image.open(ruta).convert("RGB")
            img_t = preprocess(img).unsqueeze(0)
            with torch.no_grad():
                img_features = modelo.encode_image(img_t)
                img_features /= img_features.norm(dim=-1, keepdim=True)
                sims = (img_features @ texto_features.T).squeeze(0).tolist()
            con.execute(
                "INSERT OR REPLACE INTO deteccion_animal_en_planta (foto_id, score_insecto, score_ave, score_nada) VALUES (?,?,?,?)",
                (foto_id, sims[0], sims[1], sims[2]),
            )
            procesadas += 1
        except Exception as e:
            errores += 1
            print(f"error en foto {foto_id} ({ruta}): {e}")
        if i % 50 == 0:
            con.commit()
            print(f"  {i}/{len(candidatas)} (ok={procesadas}, errores={errores})")
    con.commit()
    print(f"\nListo. procesadas={procesadas}, errores={errores}")

## Siguiente paso (no construido todavia)

Revisar `deteccion_animal_en_planta` ordenado por `score_insecto`/`score_ave` descendente, decidir un umbral de confianza con Remi, y para las que pasen el umbral duplicar la fila en `fotos` (mismo `ruta_original`, `ruta_actual` y `especie_id` distintos) para que entren a su propio flujo de identificacion de animal.